# Clinical Analytics Chatbot
**Drug R&D Assistant — Dataiku Notebook UI**

Run the cells below in order. The chatbot UI will appear at the bottom of the last cell.

**Before running:** set your LLM Mesh connection ID in Cell 2.

The `backend/` and `config/` folders live under `lib/python/` and are automatically
on the Python path in any Dataiku notebook or recipe in this project.

In [ ]:
# Cell 1 — Initialize Bokeh for inline notebook rendering
# No sys.path manipulation needed: Dataiku automatically adds lib/python/ to the path.
from bokeh.io import output_notebook
output_notebook()

In [ ]:
# Cell 2 — Configuration
# Replace with your actual Dataiku LLM Mesh connection ID
LLM_CONNECTION_ID = "YOUR_LLM_CONNECTION_ID"

In [ ]:
# Cell 3 — Initialize backend
import yaml
from pathlib import Path
import backend  # imported from lib/python/backend/ via Dataiku's automatic path

# Locate config relative to the backend package (lib/python/config/)
config_dir = Path(backend.__file__).parent.parent / "config"
with open(config_dir / "llm_config.yaml") as f:
    config = yaml.safe_load(f)

# Override connection ID from Cell 2
config["llm_mesh"]["connection_id"] = LLM_CONNECTION_ID

from backend.state.session_store import SessionStore
from backend.orchestrator.orchestrator import Orchestrator

session_store = SessionStore(timeout_minutes=60)
orchestrator = Orchestrator(session_store, config=config)

print("Backend initialized successfully.")

In [ ]:
# Cell 4 — Chatbot UI
import uuid
import re
import ipywidgets as widgets
from IPython.display import display, HTML
from bokeh.io import show


class ChatbotUI:
    """ipywidgets-based chat UI for use inside a Dataiku notebook."""

    # ------------------------------------------------------------------ #
    # Styles
    # ------------------------------------------------------------------ #
    _CSS = """
    <style>
      .chatbot-root { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
      .chat-header  { background:#1a1a2e; color:#fff; padding:12px 16px;
                      border-radius:10px 10px 0 0; font-size:16px; font-weight:700; }
      .chat-header small { color:#90a4ae; font-weight:400; font-size:12px; margin-left:8px; }
      .msg-user { text-align:right; margin:6px 0; }
      .msg-user span { background:#1565c0; color:#fff; padding:8px 14px;
                       border-radius:14px 14px 4px 14px; display:inline-block;
                       max-width:75%; white-space:pre-wrap; word-break:break-word; }
      .msg-bot  { text-align:left;  margin:6px 0; }
      .msg-bot  div { background:#fff; color:#1a1a2e; padding:10px 14px;
                      border-radius:14px 14px 14px 4px; display:inline-block;
                      max-width:80%; box-shadow:0 1px 3px rgba(0,0,0,.10);
                      white-space:pre-wrap; word-break:break-word; }
      .msg-bot div strong { color:#0d47a1; }
      .msg-bot div em     { color:#546e7a; }
      .section-label { font-size:11px; font-weight:700; text-transform:uppercase;
                        letter-spacing:.8px; color:#78909c; margin:10px 0 4px; }
      table.result-tbl  { border-collapse:collapse; font-size:12px; width:100%;
                           background:#fff; border-radius:8px; overflow:hidden;
                           box-shadow:0 1px 3px rgba(0,0,0,.08); }
      table.result-tbl th { background:#1565c0; color:#fff; padding:7px 10px;
                             text-align:left; font-size:11px; text-transform:uppercase;
                             white-space:nowrap; }
      table.result-tbl td { padding:6px 10px; border-bottom:1px solid #e8eaf6; vertical-align:top; }
      table.result-tbl tr:last-child td { border-bottom:none; }
      table.result-tbl tr:nth-child(even) td { background:#f8f9fd; }
      .favorable  { color:#2e7d32; font-weight:700; }
      .uncertain  { color:#e65100; font-weight:700; }
      .challenging{ color:#c62828; font-weight:700; }
    </style>
    """

    def __init__(self, orch):
        self.orch = orch
        self.session_id = str(uuid.uuid4())
        self._pending_result_id = None
        self._build_widgets()

    # ------------------------------------------------------------------ #
    # Widget construction
    # ------------------------------------------------------------------ #
    def _build_widgets(self):
        self.out = widgets.Output(
            layout=widgets.Layout(
                border='1px solid #e0e0e0',
                min_height='340px',
                max_height='520px',
                overflow_y='auto',
                padding='12px',
                background_color='#f0f2f5',
            )
        )

        self.msg_input = widgets.Textarea(
            placeholder='Describe what you need…',
            rows=2,
            layout=widgets.Layout(flex='1', min_width='0'),
        )

        self.send_btn = widgets.Button(
            description='Send',
            button_style='primary',
            layout=widgets.Layout(width='80px', height='58px'),
        )
        self.send_btn.on_click(self._on_send)

        self.upload_cro = widgets.FileUpload(
            description='CRO Sites',
            accept='.csv,.xlsx,.xls',
            multiple=False,
            layout=widgets.Layout(width='160px'),
        )
        self.upload_sponsor = widgets.FileUpload(
            description='Sponsor Sites',
            accept='.csv,.xlsx,.xls',
            multiple=False,
            layout=widgets.Layout(width='160px'),
        )
        self.upload_cro.observe(self._on_upload_cro, names='value')
        self.upload_sponsor.observe(self._on_upload_sponsor, names='value')
        self.upload_status = widgets.HTML(value='')

        self.confirm_yes = widgets.Button(
            description='\u2713 Yes, proceed',
            button_style='success',
            layout=widgets.Layout(width='150px'),
        )
        self.confirm_no = widgets.Button(
            description='\u2717 Cancel',
            button_style='danger',
            layout=widgets.Layout(width='100px'),
        )
        self.confirm_yes.on_click(self._on_confirm_yes)
        self.confirm_no.on_click(self._on_confirm_no)
        self.confirm_bar = widgets.HBox(
            [self.confirm_yes, self.confirm_no],
            layout=widgets.Layout(display='none', padding='6px 0'),
        )

        self.export_name = widgets.Text(
            placeholder='Dataiku dataset name…',
            layout=widgets.Layout(width='220px'),
        )
        self.export_btn = widgets.Button(
            description='Export to Dataset',
            button_style='info',
            layout=widgets.Layout(width='160px'),
        )
        self.export_btn.on_click(self._on_export)
        self.export_status = widgets.HTML(value='')
        self.export_bar = widgets.VBox(
            [
                widgets.HTML('<div class="section-label">Export last result to Dataiku dataset</div>'),
                widgets.HBox([self.export_name, self.export_btn]),
                self.export_status,
            ],
            layout=widgets.Layout(display='none', padding='6px 0'),
        )

    # ------------------------------------------------------------------ #
    # Layout & display
    # ------------------------------------------------------------------ #
    def display(self):
        header = widgets.HTML(
            '<div class="chat-header">'
            'Clinical Analytics Assistant'
            '<small>Drug R&amp;D | Dataiku Notebook</small>'
            '</div>'
        )
        upload_section = widgets.VBox([
            widgets.HTML('<div class="section-label">Site List Files (for Site List Merger)</div>'),
            widgets.HBox([self.upload_cro, self.upload_sponsor]),
            self.upload_status,
        ], layout=widgets.Layout(
            border='1px solid #e0e0e0', padding='8px 12px',
            background_color='#fafafa', margin='0 0 4px 0',
        ))
        input_row = widgets.HBox(
            [self.msg_input, self.send_btn],
            layout=widgets.Layout(
                border='1px solid #e0e0e0', padding='8px',
                background_color='#ffffff',
            )
        )
        root = widgets.VBox(
            [widgets.HTML(self._CSS), header, upload_section,
             self.out, self.confirm_bar, input_row, self.export_bar],
            layout=widgets.Layout(
                max_width='900px', border='1px solid #ccc',
                border_radius='10px', overflow='hidden',
            )
        )
        display(root)
        self._bot_msg(
            "Hello! I'm your Clinical Analytics Assistant. I can help you with:\n\n"
            "1. **Site List Merger** \u2014 Upload CRO and sponsor site lists to merge them\n"
            "2. **Trial Benchmarking** \u2014 Benchmark trials by indication, age group, and phase\n"
            "3. **Drug Reimbursement** \u2014 Assess reimbursement outlook by country\n"
            "4. **Enrollment Forecasting** \u2014 Generate pessimistic / moderate / optimistic enrollment curves\n\n"
            "What would you like to do?"
        )

    # ------------------------------------------------------------------ #
    # Event handlers
    # ------------------------------------------------------------------ #
    def _on_send(self, _btn):
        text = self.msg_input.value.strip()
        if not text:
            return
        self.msg_input.value = ''
        self._user_msg(text)
        self.send_btn.disabled = True
        try:
            resp = self.orch.process_message(self.session_id, text)
            self._handle_response(resp)
        finally:
            self.send_btn.disabled = False

    def _on_confirm_yes(self, _btn):
        self.confirm_bar.layout.display = 'none'
        self.send_btn.disabled = True
        try:
            resp = self.orch.handle_confirmation(self.session_id, confirmed=True)
            self._handle_response(resp)
        finally:
            self.send_btn.disabled = False

    def _on_confirm_no(self, _btn):
        self.confirm_bar.layout.display = 'none'
        resp = self.orch.handle_confirmation(self.session_id, confirmed=False)
        self._handle_response(resp)

    def _on_upload_cro(self, change):
        self._handle_upload(change, 'cro_file', 'CRO')

    def _on_upload_sponsor(self, change):
        self._handle_upload(change, 'sponsor_file', 'Sponsor')

    def _handle_upload(self, change, file_key, label):
        if not change['new']:
            return
        file_info = next(iter(change['new'].values()))
        content = file_info.get('content') or file_info.get('file_content', b'')
        filename = file_info.get('name', 'upload')

        from backend.agents.site_list_merger_agent import parse_uploaded_file
        from backend.state.conversation_state import FSMState

        class _FakeFileStorage:
            def __init__(self, name, data):
                self.filename = name
                self._data = data if isinstance(data, bytes) else bytes(data)
            def read(self):
                return self._data

        try:
            parsed = parse_uploaded_file(_FakeFileStorage(filename, content))
            state = self.orch.session_store.get_or_create(self.session_id)
            state.uploaded_files[file_key] = parsed
            state.active_skill = 'site_list_merger'
            state.fsm_state = FSMState.PARAMETER_GATHERING
            self.upload_status.value = (
                f'<span style="color:#2e7d32">'
                f'\u2713 {label}: <b>{filename}</b> ({len(parsed["data"])} rows)</span>'
            )
            self._bot_msg(
                f"{label} site list uploaded: **{filename}** "
                f"({len(parsed['data'])} rows, columns: {', '.join(parsed['columns'])})"
            )
        except ValueError as e:
            self.upload_status.value = f'<span style="color:#c62828">\u2717 {e}</span>'

    def _on_export(self, _btn):
        dataset_name = self.export_name.value.strip()
        if not dataset_name:
            self.export_status.value = '<span style="color:#c62828">Please enter a dataset name.</span>'
            return
        if not self._pending_result_id:
            self.export_status.value = '<span style="color:#c62828">No result available to export.</span>'
            return
        resp = self.orch.export_to_dataset(self.session_id, self._pending_result_id, dataset_name)
        self.export_status.value = f'<span style="color:#2e7d32">{resp.get("message", "Done.")}</span>'

    # ------------------------------------------------------------------ #
    # Response handling
    # ------------------------------------------------------------------ #
    def _handle_response(self, resp):
        if resp.get('message'):
            self._bot_msg(resp['message'])
        if resp.get('table_data'):
            self._render_table(resp['table_data'], resp.get('table_columns'))
        if resp.get('chart_json'):
            self._render_chart(resp['chart_json'])
        fsm = resp.get('fsm_state', 'idle')
        self.confirm_bar.layout.display = '' if fsm == 'confirmation_pending' else 'none'
        if resp.get('result_id'):
            self._pending_result_id = resp['result_id']
            self.export_bar.layout.display = ''
        else:
            self.export_bar.layout.display = 'none'

    # ------------------------------------------------------------------ #
    # Rendering helpers
    # ------------------------------------------------------------------ #
    def _user_msg(self, text):
        with self.out:
            display(HTML(f'<div class="msg-user"><span>{self._escape(text)}</span></div>'))

    def _bot_msg(self, text):
        with self.out:
            display(HTML(f'<div class="msg-bot"><div>{self._md(text)}</div></div>'))

    def _render_table(self, rows, columns):
        if not rows:
            return
        if not columns:
            columns = list(rows[0].keys()) if rows else []
        header_html = ''.join(f'<th>{self._escape(str(c))}</th>' for c in columns)
        rows_html = ''
        for row in rows:
            cells = ''
            for col in columns:
                val = row.get(col, '') if isinstance(row, dict) else ''
                val_str = str(val) if val is not None else ''
                low = val_str.lower()
                cls = ''
                if low == 'favorable':    cls = ' class="favorable"'
                elif low == 'uncertain':  cls = ' class="uncertain"'
                elif low == 'challenging':cls = ' class="challenging"'
                cells += f'<td{cls}>{self._escape(val_str)}</td>'
            rows_html += f'<tr>{cells}</tr>'
        table_html = (
            f'<div style="overflow-x:auto;margin:8px 0">'
            f'<table class="result-tbl">'
            f'<thead><tr>{header_html}</tr></thead>'
            f'<tbody>{rows_html}</tbody>'
            f'</table></div>'
        )
        with self.out:
            display(HTML(table_html))

    def _render_chart(self, figure_obj):
        with self.out:
            show(figure_obj)

    # ------------------------------------------------------------------ #
    # Text utilities
    # ------------------------------------------------------------------ #
    @staticmethod
    def _escape(text):
        return text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

    @classmethod
    def _md(cls, text):
        escaped = cls._escape(text)
        escaped = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', escaped)
        escaped = re.sub(r'\*(.+?)\*',     r'<em>\1</em>',         escaped)
        escaped = re.sub(r'`(.+?)`',        r'<code>\1</code>',     escaped)
        escaped = re.sub(r'^(\s*[-*]\s+)(.*)$', r'&bull; \2', escaped, flags=re.MULTILINE)
        return escaped.replace('\n', '<br/>')


print('ChatbotUI class defined. Run the next cell to launch.')

In [ ]:
# Cell 5 — Launch the chatbot
chatbot = ChatbotUI(orchestrator)
chatbot.display()